# Mean fitness from a non-equilibrium start

The population starts away from equilibrium and relaxes toward it; the mean
fitness approaches the analytic Perron eigenvalue $\bar{X}$.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

REPO = Path.cwd()
while not (REPO / "pyproject.toml").exists() and REPO != REPO.parent:
    REPO = REPO.parent

from propgen import load_summary
from propgen.plotting import set_paper_style

set_paper_style()
FIGDIR = REPO / "figures"
FIGDIR.mkdir(exist_ok=True)

summary = load_summary(REPO / "results" / "meanfit" / "summary.npz")
panel = summary.select()
mean, sem, cycles = panel["mean"], panel["sem"], panel["cycles"]
f_eq = panel["f_eq"].reshape(summary.shape)
summary

In [ ]:
repro_probs = np.array([0.009, 0.002])
Ng, Np = summary.shape

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for g in range(Ng):
    for p in range(Np):
        line, = axes[0].plot(cycles, mean[g, p], lw=2, label=f"$g$={g}, $p$={p}")
        axes[0].fill_between(cycles, mean[g, p] - sem[g, p], mean[g, p] + sem[g, p],
                             color=line.get_color(), alpha=0.2)
        axes[0].axhline(f_eq[g, p], color=line.get_color(), ls="--", lw=2, alpha=0.4)
axes[0].set_xlabel("Dilution cycle")
axes[0].set_ylabel("Frequency")
axes[0].set_title("Genotype-phenotype frequencies")
axes[0].legend(fontsize=11)

mean_fitness = np.einsum("gpt,p->t", mean, repro_probs)
axes[1].plot(cycles, mean_fitness, lw=2, color="black")
axes[1].axhline(panel["Xbar_theory"], ls="--", lw=2, color="red",
                label=rf"$\bar{{X}}$ = {panel['Xbar_theory']:.5g}")
axes[1].set_xlabel("Dilution cycle")
axes[1].set_ylabel("Mean fitness")
axes[1].set_title("Mean fitness")
axes[1].legend()

plt.tight_layout()
plt.savefig(FIGDIR / "meanfit.pdf")
plt.show()